In [40]:
!pip list |egrep -i 'kubernetes|kserve'

kfp-kubernetes                     1.4.0
kserve                             0.15.2
kubernetes                         26.1.0


In [50]:
import kfp
from kfp import dsl
from kfp import kubernetes
from kfp import local
from kfp.dsl import Input, Output, Dataset, Model, Artifact

# TIP: you may need to authenticate with the KFP instance
# local.init(runner=local.SubprocessRunner())
kfp_client = kfp.Client()

In [51]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9",
)
def create_secret(namespace: str,secret_name: str) -> str:
    import os
    import base64
    import kserve
    from kubernetes import client,config,utils
    from kubernetes.client.rest import ApiException

    config.load_incluster_config()
    v1 = client.CoreV1Api()
    custom_api = client.CustomObjectsApi()
    
    with open('/etc/secrets/ezua/.auth_token','r') as file:
        AUTH_TOKEN = file.read().strip()

    try:
        ## Check pre-exists secret
        preexists_secret = v1.read_namespaced_secret(namespace=namespace,name=secret_name)
        print(f"secret name : {preexists_secret.metadata.name} exists")
        v1.delete_namespaced_secret(namespace=namespace,name=secret_name)
    
    except ApiException as e:
        print("Exception when calling CoreV1Api : %s\n" % e)

    finally:
        ## Create secret snippet start
        secret_data_encoded = {
            "AWS_ACCESS_KEY_ID": base64.b64encode(AUTH_TOKEN.encode()).decode(),
            "AWS_SECRET_ACCESS_KEY": base64.b64encode("s3".encode()).decode()
        }
        
        secret_body = client.V1Secret(
            api_version="v1",
            kind="Secret",
            metadata=client.V1ObjectMeta(
                name=secret_name,
                annotations={
                    "serving.kserve.io/s3-cabundle":"",
                    "serving.kserve.io/s3-endpoint":"local-s3-service.ezdata-system.svc.cluster.local:30000/",
                    "serving.kserve.io/s3-useanoncredential":"false",
                    "serving.kserve.io/s3-usehttps":"0",
                    "serving.kserve.io/s3-verifyssl":"0",
                }
            ),
            type="Opaque",
            data=secret_data_encoded,  # Use data or string_data
        )
        
        api_response = v1.create_namespaced_secret(namespace=namespace, body=secret_body)
        print(f"Secret '{api_response.metadata.name}' created successfully in namespace '{namespace}'.\n api_response:{api_response}")
        ## Create secret snippet end
        return api_response.metadata.name

In [52]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9",
)
def create_service_account(namespace: str,secret_name: str,service_account_name: str) -> str:
    import kserve
    from kubernetes import client,config,utils
    from kubernetes.client.rest import ApiException

    config.load_incluster_config()
    v1 = client.CoreV1Api()
    custom_api = client.CustomObjectsApi()
    
    try:
        ## Check pre-exists serviceaccount
        preexists_sa = v1.read_namespaced_service_account(namespace=namespace,name=service_account_name)
        print(f"Service Account name : {preexists_sa.metadata.name} exists")
        v1.delete_namespaced_service_account(namespace=namespace,name=service_account_name)
    
    except ApiException as e:
        print("Exception when calling CoreV1Api : %s\n" % e)

    finally:
        ## Create Service Account snippet start
        service_account_body = client.V1ServiceAccount(
            api_version="v1",
            kind="ServiceAccount",
            metadata=client.V1ObjectMeta(
                name=service_account_name, 
                namespace=namespace
            ),
            secrets=[
                client.V1ObjectReference(name=secret_name)
            ]
        )
        api_response = v1.create_namespaced_service_account(
            namespace=namespace,
            body=service_account_body
        )
        print(f"ServiceAccount '{api_response.metadata.name}' created successfully in namespace '{namespace}'.")
        ## Create Service Account snippet end
        return api_response.metadata.name

In [53]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9", 
)
def create_serving_runtime(namespace: str,runtime_name: str) -> str:
    import kserve
    from kubernetes import client,config,utils
    from kubernetes.client.rest import ApiException
    from kserve import V1alpha1ServingRuntime, V1alpha1ServingRuntimeSpec, V1alpha1SupportedModelFormat

    config.load_incluster_config()
    custom_api = client.CustomObjectsApi()

    try:
        ## Check pre-exists ServingRuntime
        preexists_runtime = custom_api.get_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1alpha1",
            namespace=namespace,
            plural="servingruntimes",
            name=runtime_name
        )

        print(f"Serving Runtime name : {preexists_runtime['metadata']['name']} exists")
        custom_api.delete_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1alpha1",
            namespace=namespace,
            plural="servingruntimes",
            name=runtime_name
        )
        
    except ApiException as e:
        print("Exception when calling CoreV1Api : %s\n" % e)

    finally:
        ## Create Serving Runtime snippet start
        runtime_body = V1alpha1ServingRuntime(
            api_version='serving.kserve.io/v1alpha1',
            kind='ServingRuntime',
            metadata=client.V1ObjectMeta(
                name=runtime_name,
            ),
            spec=V1alpha1ServingRuntimeSpec(
                annotations={
                    "prometheus.kserve.io/path":"/metrics",
                    "prometheus.kserve.io/port":"8002",
                },
                containers=[
                    client.V1Container(
                        args=['tritonserver','--model-store=/mnt/models','--grpc-port=9000','--http-port=8080','--allow-grpc=true','--allow-http=true'],
                        image='nvcr.io/nvidia/tritonserver:25.02-py3',
                        name='kserve-container',
                        resources=client.V1ResourceRequirements(
                            limits={
                                "cpu": "1",
                                "memory": "2Gi"
                            },
                            requests={
                                "cpu": "1",
                                "memory": "2Gi"
                            }
                        )
                    )
                ],
                protocol_versions=['v2','grpc-v2'],
                supported_model_formats=[
                    V1alpha1SupportedModelFormat(auto_select=True,name='tensorrt',version='8'),
                    V1alpha1SupportedModelFormat(auto_select=True,name='triton',version='2'),
                ]
            )
        )
    
        api_response = custom_api.create_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1alpha1",
            namespace=namespace,
            plural="servingruntimes",
            body=runtime_body,
        )
        print(f"Serving Runtime created : {api_response['metadata']['name']}")
        ## Create servingruntime snippet end 
        return api_response['metadata']['name']

In [54]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9",
)
def create_isvc(namespace: str,service_account_name:str,runtime_name:str,isvc_name:str,model_s3_url:str,num_of_gpus:str) -> str:
    import time
    import kserve
    from kubernetes import client,config,utils
    from kubernetes.client.rest import ApiException
    from kserve import V1beta1InferenceService, V1beta1InferenceServiceSpec, V1beta1PredictorSpec,V1beta1ModelSpec,V1beta1ModelFormat


    config.load_incluster_config()
    custom_api = client.CustomObjectsApi()

    try:
        ## Check pre-exists Inference Service
        preexists_isvc = custom_api.get_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=namespace,
            plural="inferenceservices",
            name=isvc_name,
        )
        print(f"Inference Service name : {preexists_isvc['metadata']['name']} exists")
        custom_api.delete_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=namespace,
            plural="inferenceservices",
            name=isvc_name,
        )
        time.sleep(3)
    except ApiException as e:
        print("Exception when calling CoreV1Api : %s\n" % e)

    finally:
        ## Create Inference Service snippet start
        isvc_body = V1beta1InferenceService(
            api_version='serving.kserve.io/v1beta1',
            kind='InferenceService',
            metadata=client.V1ObjectMeta(
                name=isvc_name,
            ),
            spec=V1beta1InferenceServiceSpec(
                predictor=V1beta1PredictorSpec(
                    service_account_name=service_account_name,
                    model=V1beta1ModelSpec(
                        model_format=V1beta1ModelFormat(
                            name="triton"
                        ),
                        storage_uri=model_s3_url,
                        runtime=runtime_name,
                        resources=client.V1ResourceRequirements(
                            limits={
                                "nvidia.com/gpu": num_of_gpus,
                            },
                            requests={
                                "nvidia.com/gpu": num_of_gpus,
                            }
                        )
                    )
                )
            )
        )
        
        
        api = client.CustomObjectsApi()
        api.create_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=namespace,
            plural="inferenceservices",
            body=isvc_body,
        )
        print("Inference Service created")
        time.sleep(3)
        while True:
            created_isvc = api.get_namespaced_custom_object(
                group="serving.kserve.io",
                version="v1beta1",
                namespace=namespace,
                plural="inferenceservices",
                name=isvc_name,
            )
            if created_isvc is not None:
                if created_isvc.get('status') is not None: 
                    if created_isvc.get('status').get('address') is not None:
                        url = created_isvc.get('status').get('address').get('url')
                        print(f"{url} is ready")
                        return url
            print(f"url is not ready for {created_isvc['metadata']['name']}")
            time.sleep(3)
        ## Create servingruntime snippet end

In [55]:
@dsl.component(
    base_image="geuntakroh/kfp-test:v0.9",
)
def sample_inference(isvc_url: str, vehicle_detection_result: Output[Artifact],license_detection_result: Output[Artifact]):
    from ultralytics import YOLO
    import subprocess
    from urllib.parse import urlparse
    import os
    
    parts = urlparse(isvc_url)
    svc_name = parts.netloc.split('.')[0]
    namespace = parts.netloc.split('.')[1]
    url_for_yolo = f"http://{svc_name}-00001.{namespace}.svc.cluster.local"

    vehicle_detector = YOLO(f"{url_for_yolo}/vehicle_detector",task='detect')
    license_detector = YOLO(f"{url_for_yolo}/license_detector",task='detect')

    vehicle_results = vehicle_detector("./Samples/multiple_cars.png",save=True)
    vehicle_output_file = vehicle_results[0].save_dir + '/' + os.listdir(vehicle_results[0].save_dir)[0]
    print(vehicle_output_file)
    license_results = license_detector("./Samples/multiple_cars.png",save=True)
    license_output_file = license_results[0].save_dir + '/' + os.listdir(license_results[0].save_dir)[0]
    print(license_output_file)

    subprocess.run(['cp',vehicle_output_file,vehicle_detection_result.path ])    
    subprocess.run(['cp',license_output_file,license_detection_result.path ])    

In [60]:
@dsl.pipeline()
def create_inference_service_pipeline(
    # AUTH_TOKEN: str,
    namespace: str,
    secret_name: str,
    service_account_name: str,
    runtime_name: str,
    isvc_name:str,
    model_s3_url:str,
    num_of_gpus:str
):
    with open('/etc/secrets/ezua/.auth_token','r') as file:
        AUTH_TOKEN = file.read()
    task1 = create_secret(namespace=namespace,secret_name=secret_name)
    kfp.kubernetes.add_pod_annotation(
        task=task1,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    
    task2 = create_service_account(namespace=namespace,secret_name=task1.output,service_account_name=service_account_name)
    kfp.kubernetes.add_pod_annotation(
        task=task2,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    
    task3 = create_serving_runtime(namespace=namespace,runtime_name=runtime_name)
    kfp.kubernetes.add_pod_annotation(
        task=task3,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    
    task4 = create_isvc(namespace=namespace,service_account_name=task2.output,runtime_name=task3.output,isvc_name=isvc_name,model_s3_url=model_s3_url,num_of_gpus=num_of_gpus)
    kfp.kubernetes.add_pod_annotation(
        task=task4,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    
    task5 = sample_inference(isvc_url=task4.output)
    kfp.kubernetes.add_pod_annotation(
        task=task5,
        annotation_key="hpe-ezua/add-auth-token",
        annotation_value="true"
    )
    task5.set_cpu_limit("4")
    task5.set_memory_limit("16Gi")

In [61]:
current_sc = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.spec.storageClassName}'").read()
namespace_cur = os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
print(namespace_cur)
print(current_sc)

geun-tak-roh-2e590eb8
gl4f-filesystem


In [62]:
namespace=os.popen("kubectl get pvc user-pvc -o=jsonpath='{.metadata.namespace}'").read()
secret_name = "my-secret"
service_account_name = "my-service-account"
runtime_name = "my-ngc"
isvc_name = "my-isvc"
model_s3_url='s3://mlflow.sg2pcai172/7/f0cacf2d97a54b70bfc7e3499a400f99/artifacts' + '/triton/triton_engines'
num_of_gpus="1"

kfp_client.create_run_from_pipeline_func(
    create_inference_service_pipeline,
    arguments={
        # 'AUTH_TOKEN': AUTH_TOKEN
        'namespace': namespace,
        'secret_name': secret_name,
        'service_account_name': service_account_name,
        'runtime_name': runtime_name,
        'isvc_name': isvc_name,
        'model_s3_url': model_s3_url,
        'num_of_gpus': num_of_gpus
    },
    experiment_name="test-rhgt-exp",
    enable_caching=False # failed: failed to create PVC and publish execution createpvc: failed to create cache entrty for create pvc: failed to create task: rpc error: code = InvalidArgument desc = Failed to create a new task due to validation error: Invalid input error: Invalid task: must specify FingerPrint
    # enable_caching=True
)

RunPipelineResult(run_id=4f53ab05-6349-4bbd-9abd-fc62bbe1ae9b)

In [59]:
from kfp import compiler, dsl

compiler.Compiler().compile(create_inference_service_pipeline, package_path='create_inference_service_pipeline.yaml')